# Square hard-wall Chern dynamics: ten random centers

Nx=Ny=L=20/30/40; 100 pure trajectories each, exactly 40 cycles (observations 0–40). Perfect correction, hard support truncation, slab-only raster-y measurements, alpha1=1, alpha2=30, nshell=1, periodic boundaries, complex128. Cycle zero is after canonical exterior preparation.

At each cycle and for each trajectory, choose ten distinct random y centers, with x=L/2 and R=0.2L (4, 6, 8). Interfaces are (5,15), (8,22), (10,30). Periodic wrapping preserves disks crossing y=0. Compute the three-sector estimator from Gamma=(V V†)^T. Save every center individually and average centers within each trajectory before forming the trajectory mean and standard error. No covariance history.

The first run benchmarks ascending batch sizes 5/10/25/50/100 (2/1 fallback), stopping growth at a measured limit, using the actual observer. Qualified means <32 GiB peak reserved and <45 minutes forecast including 25% margin. Calibration is not production data. There is no prequalified batch size. Completed batches resume; an interrupted batch restarts. Use only one writer.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

# All scientific and execution settings are visible here. This revision's
# scientific/calibration contract is fixed; edits require a new revision.
REPORT_ONLY = False
MAX_NEW_BATCHES = None  # e.g. 1 to stop after one production batch
BUNDLE_DRIVE_DIR = Path('/content/drive/MyDrive/final_production_new_designs/27_square_hard_wall_random_center_chern')
LOCAL_BUNDLE_DIR = Path('/content/27_square_hard_wall_random_center_chern')
SCRATCH_ROOT = Path('/content/square_random_center_chern_scratch')
CONFIG = {'sampling_revision': 'square_hard_wall_random_center_chern_l20-30-40_s100_t40_r0p2_v1',
 'root_seed': 2026092801,
 'sizes': [20, 30, 40],
 'samples': 100,
 'cycles': 40,
 'alpha_1': 1.0,
 'alpha_2': 30.0,
 'nshell': 1,
 'DW': True,
 'dw_truncation': True,
 'triv_region_local_mode': False,
 'meas_slab_only': True,
 'perfect_correction': True,
 'postselect': False,
 'postselect_probability': 0.0,
 'sequence': 'raster_y',
 'init_mode': 'default',
 'filling_frac': 0.5,
 'n_a': 0.5,
 'trial_orbitals': 'X',
 'boundary_conditions': 'periodic',
 'dtype': 'complex128',
 'backend': 'local',
 'state_representation': 'physical_frame',
 'frame_reorthonormalize_interval': 1,
 'centers_per_cycle': 10,
 'radius_fraction': 0.2,
 'center_chunk_size': 10,
 'calibration_batches': [5, 10, 25, 50, 100],
 'fallback_batches': [2, 1],
 'warmup_cycles': 1,
 'timed_cycles': 5,
 'memory_limit_gib': 32.0,
 'forecast_margin': 1.25,
 'max_forecast_minutes': 45.0,
 'stop_after_batch_seconds': 3600.0}
OUTPUT_ROOT = Path('/content/drive/MyDrive/classA_final_production_outputs') / CONFIG['sampling_revision']


## Stage locally and run/resume

Prints source/config identity, measured timings, memory, frozen batch sizes and remaining-time forecast. Both stderr and stdout are relayed, including tqdm carriage-return updates. A saved batch longer than one hour stops the queue. DriveFS readback is checked; it is not independent server-side verification.


In [ ]:
import codecs
import json
import os
import shutil
import subprocess
import sys

LOCAL_BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
for relative in ['run_campaign.py', 'random_center_observer.py', 'src/classA_U1FGTN_gpu.py', 'src/occupied_frame_gpu.py']:
    target = LOCAL_BUNDLE_DIR / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(BUNDLE_DRIVE_DIR / relative, target)
config_path = LOCAL_BUNDLE_DIR / 'resolved_config.json'
config_path.write_text(json.dumps(CONFIG, indent=2))
command = [sys.executable, '-u', str(LOCAL_BUNDLE_DIR / 'run_campaign.py'),
           '--config', str(config_path), '--output-root', str(OUTPUT_ROOT),
           '--scratch-root', str(SCRATCH_ROOT)]
if REPORT_ONLY:
    command.append('--report-only')
if MAX_NEW_BATCHES is not None:
    command += ['--max-new-batches', str(MAX_NEW_BATCHES)]
print('[launch]', ' '.join(command), flush=True)
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
try:
    while True:
        block = os.read(process.stdout.fileno(), 4096)
        if not block:
            break
        sys.stdout.write(decoder.decode(block))
        sys.stdout.flush()
    sys.stdout.write(decoder.decode(b'', final=True))
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, command)
finally:
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
print('[notebook] requested session finished', flush=True)


## Preview: center-averaged trajectories

Verified completed batches only; labels report actual S. Shading is std(c_xi, ddof=1)/sqrt(S), not an error over individual centers.


In [ ]:
import importlib.util
import matplotlib.pyplot as plt
sys.path.insert(0, str(LOCAL_BUNDLE_DIR))
sys.modules.pop('random_center_observer', None)
spec = importlib.util.spec_from_file_location('chern_preview', LOCAL_BUNDLE_DIR / 'run_campaign.py')
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
products = runner.preview(OUTPUT_ROOT)
fig, ax = plt.subplots(figsize=(5, 3.3))
for ny, row in products.items():
    line, = ax.plot(row['cycles'], row['mean'], label=f'{ny} x {ny}, S={row["samples"]}')
    ax.fill_between(row['cycles'], row['mean']-row['sem'], row['mean']+row['sem'], alpha=.2, color=line.get_color())
ax.axhline(1, color='gray', ls='--', lw=.8)
ax.set(xlabel='cycle', ylabel='center-averaged Chern number')
ax.tick_params(direction='in', top=True, right=True)
if products:
    ax.legend(frameon=False)
else:
    print('No verified production batches yet.')
fig.tight_layout()
plt.show()


In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')
